# deep56 lat_iso i1: running the pISO pipeline

Every stage, from raw maps to TT spectra with analytic error bars, on two arrays (i1 f090 and f150), so three spectra: two autos and the cross. Each step prints an `sbatch` command to run on the cluster; run the cells below it afterwards to look at what it wrote.

Before starting, copy `paramfiles/deep56_20260201_skn_50cg_inpaint_holes.dict`, set `data_dir`, `yaml_dir` and `arrays_lat_iso = ['i1_f090', 'i1_f150']`, and point `PARAMFILE` below at the copy.

| Step | Script | Gives |
|---|---|---|
| 0c | `masks/get_xtra_mask.py` | where we observed deeply enough |
| 1 | `get_windows.py` | the sky footprint |
| 2 | `get_best_fit_mflike.py` | theory to compare against |
| 3 | `get_mcm_bbl_and_pseudosignal.py` | how the window mixes multipoles |
| 4 | `get_pseudo2datavec.py` | the multiplicative steps as one matrix |
| 5 | `get_spectra_from_maps.py` | the spectrum |
| 6 | `get_pseudonoise.py` | the noise level |
| 7 | `get_w2_w4_and_window_spectra.py` | window geometry for the covariance |
| 7b | `get_ana_cov_blocks_recipe.py` | which covariance blocks to build |
| 8 | `get_ana_cov_blocks.py` | the error bars |
| 9 | `get_fg_covariance_blocks.py` | foreground contribution to the errors |

Most stages use MPI and are launched with `srun`; `masks/get_xtra_mask.py` and `get_best_fit_mflike.py` are not, and the `sbatch()` helper takes care of this. `--ntasks` must not exceed the number of spectra.

In [ ]:
import os
from os.path import join as opj

import numpy as np
import matplotlib.pyplot as plt
from pixell import enmap

from pspy import so_dict, so_map, so_spectra

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

In [ ]:
REPO = os.path.abspath("..")          # project/SO/pISO, when run from the notebooks directory
PARAMFILE = opj(REPO, "paramfiles/deep56_20260201_skn_50cg_inpaint_holes.dict")   # a copy, see above
SLURM = opj(REPO, "slurm/tiger.slurm")
PY = opj(REPO, "python")

d = so_dict.so_dict()
d.read_from_file(PARAMFILE)

sv = "lat_iso"
arrays = d[f"arrays_{sv}"]
lmax, spec_type = d["lmax"], d["type"]
spectra = ["TT", "TE", "TB", "ET", "BT", "EE", "EB", "BE", "BB"]

# every unique array pair: N arrays -> N(N+1)/2 spectra
pairs = [(a, b) for i, a in enumerate(arrays) for b in arrays[i:]]
spec_names = [f"{sv}_{a}x{sv}_{b}" for a, b in pairs]
NTASKS = len(spec_names)

# the one we follow in detail through the plots below
ar = arrays[0]
spec_name = f"{sv}_{ar}x{sv}_{ar}"

mask_dir, window_dir = d["mask_dir"], d["window_dir"]
mcm_dir, spec_dir, cov_dir = d["mcm_dir"], d["spec_dir"], d["cov_dir"]
bestfit_dir, alms_dir = d["best_fits_dir"], d["alms_dir"]
noise_dir = opj(bestfit_dir, "noise")

print(f"arrays  {arrays}")
print(f"splits  {d['n_splits_lat_iso']}")
print(f"lmax    {lmax}   type {spec_type}")
print(f"output  {d['data_dir']}")
print(f"\n{len(spec_names)} spectra, so --ntasks must not exceed {len(spec_names)}:")
for s in spec_names:
    print("  ", s)

In [ ]:
# scripts that call so_mpi.init need srun; the other two must run as plain python
NO_MPI = {"masks/get_xtra_mask.py", "get_best_fit_mflike.py"}


def sbatch(step, resources, script, extra=""):
    """Print the command for a pipeline step. Paste into a terminal on tiger."""
    srun = "" if script in NO_MPI else "srun "
    if srun and "--ntasks" not in resources:
        resources = f"--ntasks {NTASKS} " + resources
    print(f"sbatch {resources} --job-name {os.path.basename(script)[:-3]} \\\n"
          f"  {SLURM} {srun}python -u {opj(PY, script)} \\\n  {PARAMFILE} {extra}".rstrip())


def show_outputs(directory, pattern=""):
    """List what a step wrote, with sizes."""
    if not os.path.exists(directory):
        print(f"missing: {directory}")
        return
    fs = sorted(f for f in os.listdir(directory) if pattern in f)
    print(f"{len(fs)} file(s) in {directory}")
    for f in fs[:12]:
        print(f"  {os.path.getsize(opj(directory, f)) / 1e6:8.1f} MB  {f}")
    if len(fs) > 12:
        print(f"  ... {len(fs) - 12} more")

---
## 0c: observation masks

Per split, keeps the pixels whose smoothed inverse variance is above the 20th percentile of the non-zero values, then intersects the splits. The same is done for the cross-linking map. Settings are in `python/yaml/xtra_mask.yaml`.

$$m_i = \Theta\!\left[\,\mathcal{S}_{0.5^\circ}(\sigma_i^{-2}) - q_{0.2}\,\right], \qquad m = \prod_i m_i$$

$\mathcal{S}$ is a Gaussian smooth, $q_{0.2}$ the 20th percentile and $\Theta$ a step function.

Writes to `mask_dir`: one mask per split, those intersected per array, then `xtra_ivar_mask_intersect.fits` and `xtra_xlink_mask_intersect.fits`, which step 1 uses. `--use-xlink` is needed because the paramfile's mask list references the cross-linking mask.

In [ ]:
sbatch("0c", "--mem 48G --cpus-per-task 4 --time 20:00",
       "masks/get_xtra_mask.py", extra="--use-xlink")

In [ ]:
show_outputs(mask_dir)

In [ ]:
# split 0 vs the intersection: a pixel must pass the depth cut in every split to survive,
# so the intersection is the tighter of the two.
to_show = [(f"xtra_ivar_mask_{sv}_{ar}_set0.fits", "ivar, split 0"),
           ("xtra_ivar_mask_intersect.fits", "ivar, intersected"),
           ("xtra_xlink_mask_intersect.fits", "cross-linking")]

fig, axs = plt.subplots(3, 1, figsize=(11, 9))
for a, (fn, title) in zip(axs, to_show):
    p = opj(mask_dir, fn)
    if not os.path.exists(p):
        a.text(0.5, 0.5, f"missing {fn}", ha="center"); a.set_axis_off(); continue
    m = enmap.read_map(p)
    a.imshow(m, origin="lower", cmap="gray", interpolation="nearest")
    a.set_title(f"{title} — keeps {np.mean(m > 0):.1%}")
    a.set_xticks([]); a.set_yticks([])
fig.tight_layout()

---
## 1: windows

A window gives each pixel a weight from 0 to 1: the masks multiplied together and apodised, since a hard edge rings in harmonic space. Two come out: **kspace** (1° taper), used when applying the k-space filter, and **baseline** (edge cut back by 1.5°, 2° taper, times the source mask), used for the spectrum.

$$W_{\rm ks} = \mathcal{A}_{1^\circ}\!\left[\prod_k M_k\right], \qquad
W_{\rm base} = \mathcal{A}_{2^\circ}\!\left[\prod_k M_k \cdot \Theta(D > 1.5^\circ)\right] \cdot M_{\rm src}$$

$\mathcal{A}_\theta$ is a C1 apodisation of width $\theta$ and $D$ the distance to the nearest masked pixel.

Writes `window_intersect_kspace.fits` and `window_intersect_baseline.fits` to `window_dir`, one pair per array.

In [ ]:
for m in d["kspace_masks_intersect"]:
    print("kspace  ", os.path.basename(m))
for m in d["baseline_masks_intersect"]:
    print("baseline", os.path.basename(m))
print(f"\napod {d['apod_kspace_degree']} deg -> kspace, "
      f"{d['apod_survey_degree']} deg -> baseline, edge cut {d['kspace_to_baseline_edge_cut']} deg")

sbatch("1", "--mem 72G --cpus-per-task 10 --time 20:00", "get_windows.py")

In [ ]:
show_outputs(window_dir)

win_base = so_map.read_map(opj(window_dir, "window_intersect_baseline.fits"))
win_ks = so_map.read_map(opj(window_dir, "window_intersect_kspace.fits"))


def solid_angle(w):
    return np.sum(w * w.pixsizemap()) / (4 * np.pi) * 41253


fig, axs = plt.subplots(2, 1, figsize=(11, 7))
for a, w, name in zip(axs, [win_ks.data, win_base.data], ["kspace", "baseline"]):
    im = a.imshow(w, origin="lower", cmap="viridis", vmin=0, vmax=1)
    a.set_title(f"{name} — {solid_angle(w):.0f} sq deg")
    a.set_xticks([]); a.set_yticks([])
    plt.colorbar(im, ax=a, fraction=0.02)
fig.tight_layout()

In [ ]:
# the taper, seen edge on
row = win_base.data.shape[0] // 2
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(win_ks.data[row], lw=1, label="kspace")
ax.plot(win_base.data[row], lw=1, label="baseline")
ax.set_xlabel("pixel"); ax.set_ylabel("window")
ax.legend(); ax.grid(alpha=0.3)

In [ ]:
# w2 normalises the spectrum, w4 the covariance -- variance is squared, so the window
# weighting enters twice. step 7 redoes this properly per mask combination.
pix = win_base.data.pixsizemap()
w2 = np.sum(win_base.data**2 * pix) / np.sum(pix)
w4 = np.sum(win_base.data**4 * pix) / np.sum(pix)
print(f"w2 {w2:.4e}\nw4 {w4:.4e}")

---
### the data

One split of one array, temperature only, before any windowing.

In [ ]:
# raw sky map, split 0 of the first array. srcfree = point sources already subtracted.
raw = so_map.read_map(d[f"maps_{sv}_{ar}"][0])
T = raw.data[0]

fig, axs = plt.subplots(3, 1, figsize=(11, 10))

im = axs[0].imshow(T, origin="lower", cmap="RdBu_r", vmin=-300, vmax=300)
axs[0].set_title(f"{ar} split 0, T [uK] — full CAR footprint")
plt.colorbar(im, ax=axs[0], fraction=0.02)

im = axs[1].imshow(win_base.data, origin="lower", cmap="viridis", vmin=0, vmax=1)
axs[1].set_title("baseline window")
plt.colorbar(im, ax=axs[1], fraction=0.02)

im = axs[2].imshow(T * win_base.data, origin="lower", cmap="RdBu_r", vmin=-300, vmax=300)
axs[2].set_title("product — this is what gets transformed")
plt.colorbar(im, ax=axs[2], fraction=0.02)

for a in axs:
    a.set_xticks([]); a.set_yticks([])
fig.tight_layout()

print(f"map shape {raw.data.shape}, T range {T.min():.0f} to {T.max():.0f} uK")
print(f"rms inside window {np.std(T[win_base.data > 0.9]):.1f} uK")

In [ ]:
# zoom on the middle. the mottling is CMB plus atmospheric/detector noise; the horizontal
# streaking is the scan direction, which is what the kspace filter is there to remove.
ny, nx = T.shape
sy, sx = slice(ny//2 - 150, ny//2 + 150), slice(nx//2 - 300, nx//2 + 300)

fig, ax = plt.subplots(figsize=(11, 5))
im = ax.imshow(T[sy, sx], origin="lower", cmap="RdBu_r", vmin=-200, vmax=200)
ax.set_title("zoom, deep interior")
ax.set_xticks([]); ax.set_yticks([])
plt.colorbar(im, ax=ax, fraction=0.03)

---
## 2: theory

CAMB for the CMB and mflike/fgspectra for the foregrounds, $D_\ell = D_\ell^{\rm CMB} + D_\ell^{\rm fg}$ with $D_\ell = \ell(\ell+1)C_\ell/2\pi$. Used for comparison and as the signal term of the covariance.

Writes `cmb.dat`, `fg_<spec>.dat` and `cmb_and_fg_<spec>.dat` to `best_fits_dir`, with a `components/` breakdown per foreground.

In [ ]:
sbatch("2", "--mem 16G --cpus-per-task 10 --time 20:00", "get_best_fit_mflike.py")

In [ ]:
show_outputs(bestfit_dir, ".dat")

bf_fn = opj(bestfit_dir, f"cmb_and_fg_{spec_name}.dat")
fg_fn = opj(bestfit_dir, f"fg_{spec_name}.dat")
l_th, ps_th = so_spectra.read_ps(bf_fn, spectra=spectra)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(l_th, ps_th["TT"], "k-", lw=1.5, label="CMB + fg")
if os.path.exists(fg_fn):
    _, ps_fg = so_spectra.read_ps(fg_fn, spectra=spectra)
    ax.plot(l_th, ps_fg["TT"], lw=1, ls="--", label="fg")
    ax.plot(l_th, ps_th["TT"] - ps_fg["TT"], lw=1, ls=":", label="CMB")
ax.set_xlim(0, 6000); ax.set_yscale("log")
ax.set_xlabel(r"$\ell$"); ax.set_ylabel(r"$D_\ell^{TT}$ [$\mu K^2$]")
ax.legend(); ax.grid(alpha=0.3)

---
## 3: mode coupling

The window mixes multipoles, $\langle \tilde{C}_\ell \rangle = \sum_{\ell'} M_{\ell\ell'} C_{\ell'}$, with

$$M_{\ell\ell'} = \frac{2\ell'+1}{4\pi}\sum_{\ell''}(2\ell''+1) W_{\ell''}
\begin{pmatrix} \ell & \ell' & \ell'' \\ 0 & 0 & 0 \end{pmatrix}^{2}$$

and $W_\ell$ the cross-power spectrum of the two windows. The matrix is binned (`binned_mcm = True`) and inverted; step 5 applies $M^{-1}$. There is one matrix per window pair, so $N(N+1)/2$ for $N$ arrays.

Writes `<spec>_mode_coupling_inv.npy`, shaped `(5, nbin, nell)` with spin blocks `(00, 02, 20, ++, --)` (TT is block 0), and `<spec>_Bbl.npy` to `mcm_dir`.

In [ ]:
sbatch("3", f"--ntasks {NTASKS} --cpus-per-task 8 --mem-per-cpu 8G --time 30:00",
       "get_mcm_bbl_and_pseudosignal.py")

In [ ]:
show_outputs(mcm_dir)

mbb_inv = np.load(opj(mcm_dir, f"{spec_name}_mode_coupling_inv.npy"))
Bbl = np.load(opj(mcm_dir, f"{spec_name}_Bbl.npy"))
nbin, nell_mcm = mbb_inv[0].shape
print(f"\nmcm {mbb_inv.shape} -> TT block is {nbin} bins x {nell_mcm} ell")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
v = np.abs(mbb_inv[0]).max() / 50
im = axs[0].imshow(mbb_inv[0], aspect="auto", origin="lower", cmap="RdBu_r", vmin=-v, vmax=v)
axs[0].set_xlabel(r"$\ell$"); axs[0].set_ylabel("bin"); axs[0].set_title(r"$M^{-1}_{00}$")
plt.colorbar(im, ax=axs[0], fraction=0.04)

# each row shows which multipoles feed one bin
for b in [5, 20, 40]:
    axs[1].plot(mbb_inv[0][b], lw=0.8, label=f"bin {b}")
axs[1].set_xlabel(r"$\ell$"); axs[1].legend(); axs[1].grid(alpha=0.3)
fig.tight_layout()

---
## 4: pseudo2datavec

Folds mode coupling, binning, beam and transfer function into one matrix, so step 5 is a single product. Step 8 needs it too.

$$\mathsf{P} = M_b^{-1} \, \mathsf{B} \, F_\ell \, B_\ell^2 \qquad\Longrightarrow\qquad D_b = \mathsf{P}\,\tilde{C}_\ell$$

with $\mathsf{B}$ the binning, $F_\ell$ the k-space transfer function and $B_\ell$ the beam ($B_\ell^{(i)}B_\ell^{(j)}$ for two arrays).

Writes `pseudo2datavec_<spec>.npy`, a dict of sparse blocks, to `mcm_dir`.

In [ ]:
sbatch("4", "--cpus-per-task 2 --mem-per-cpu 8G --time 20:00", "get_pseudo2datavec.py")

In [ ]:
show_outputs(mcm_dir, "pseudo2datavec")

---
## 5: spectra

Maps, k-space filter, window, spherical harmonic transform, spectra of split pairs, deconvolution. Three outputs per spectrum:

- **cross**: different splits, so the noise averages away. This is the signal estimate.
- **auto**: a split with itself, signal plus noise.
- **noise**: auto minus cross.

Four splits give 6 cross pairs per auto spectrum. The cross-frequency spectrum keeps the same-split pairs too (16 pairs), since different detectors have independent noise.

Writes `Dl_<spec>_{cross,auto,noise}.dat` to `spec_dir` and the alms of each split to `alms_dir`, which step 6 reads.

In [ ]:
sbatch("5", f"--ntasks {NTASKS} --mem 200G --cpus-per-task 4 --time 30:00",
       "get_spectra_from_maps.py")

In [ ]:
show_outputs(spec_dir)
print()
show_outputs(alms_dir)

lb, ps_cross = so_spectra.read_ps(opj(spec_dir, f"{spec_type}_{spec_name}_cross.dat"), spectra=spectra)
_, ps_auto = so_spectra.read_ps(opj(spec_dir, f"{spec_type}_{spec_name}_auto.dat"), spectra=spectra)
_, ps_noise = so_spectra.read_ps(opj(spec_dir, f"{spec_type}_{spec_name}_noise.dat"), spectra=spectra)
print(f"\n{len(lb)} bins, ell {lb[0]:.0f}..{lb[-1]:.0f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(l_th, ps_th["TT"], "k-", lw=1, alpha=0.6, label="theory")
ax.plot(lb, ps_auto["TT"], "^", ms=4, alpha=0.7, label="auto (signal+noise)")
ax.plot(lb, ps_cross["TT"], "o", ms=4, label="cross (signal)")
ax.plot(lb, ps_noise["TT"], "v", ms=4, alpha=0.7, label="noise")
ax.set_xlim(0, 6000); ax.set_yscale("log")
ax.set_xlabel(r"$\ell$"); ax.set_ylabel(r"$D_\ell^{TT}$ [$\mu K^2$]")
ax.set_title("where auto meets noise, the measurement has run out")
ax.legend(); ax.grid(alpha=0.3)

In [ ]:
# all three spectra. the f090 x f150 cross sits between the two autos: same CMB, but
# each auto carries its own noise while the cross carries none.
fig, ax = plt.subplots(figsize=(9, 5))
for sn in spec_names:
    _, ps = so_spectra.read_ps(opj(spec_dir, f"{spec_type}_{sn}_cross.dat"), spectra=spectra)
    label = sn.replace(f"{sv}_", "").replace(f"x{sv}_", " x ")
    ax.plot(lb, ps["TT"], "o", ms=3, label=label)

ax.plot(l_th, ps_th["TT"], "k-", lw=1, alpha=0.5, label="theory (f090)")
ax.set_xlim(0, 6000); ax.set_yscale("log")
ax.set_xlabel(r"$\ell$"); ax.set_ylabel(r"$D_\ell^{TT}$ [$\mu K^2$]")
ax.set_title("cross-split spectra for each array pair")
ax.legend(); ax.grid(alpha=0.3)

---
## 6: noise model

The noise spectrum of step 5 is itself noisy. It is smoothed with a Savitzky-Golay filter (window $w = 300$, order $k = 4$); negative bins, which appear where auto minus cross is noise dominated, are clipped to the fit.

Writes `raw_pseudo_noise_<spec>_set{k}.dat` and the smoothed `pseudo_noise_<spec>_set{k}.dat` to `best_fits_dir/noise/`, one set per array. "Pseudo" means still window-convolved; the covariance applies its own coupling later.

In [ ]:
print(f"savgol window {d['savgol_w']}, order {d['savgol_k']}")
sbatch("6", f"--ntasks {NTASKS} --mem 200G --cpus-per-task 2 --time 30:00", "get_pseudonoise.py")

In [ ]:
show_outputs(noise_dir)

---
## 7: window geometry for the covariance

$$w_2 = \frac{\int W^2 d\Omega}{\int d\Omega}, \qquad
w_4 = \frac{\int W^4 d\Omega}{\int d\Omega}, \qquad
W_\ell = \frac{1}{2\ell+1}\sum_m \left|W_{\ell m}\right|^2$$

$w_2$ normalises a spectrum and $w_4$ a covariance (the variance is quadratic, so the window weighting enters twice). Only needs step 1.

Writes `canonized_w2s.npy`, `canonized_w4s.npy`, `canonized_wls.npy` and some bookkeeping dicts to `cov_dir`. "Canonized" means equivalent mask combinations are mapped to one canonical form and computed once.

In [ ]:
sbatch("7", "--mem 120G --cpus-per-task 8 --time 40:00",
       "get_w2_w4_and_window_spectra.py")

In [ ]:
show_outputs(cov_dir, "canonized")

w2s = np.load(opj(cov_dir, "canonized_w2s.npy"), allow_pickle=True).item()
w4s = np.load(opj(cov_dir, "canonized_w4s.npy"), allow_pickle=True).item()
print(f"\nours from step 1:  w2 {w2:.4e}   w4 {w4:.4e}")
for k, v in list(w2s.items())[:3]:
    print(f"  w2 {k}: {v}")
for k, v in list(w4s.items())[:3]:
    print(f"  w4 {k}: {v}")

---
## 7b: covariance recipe

Works out which covariance blocks are needed and which four-point window combinations each one requires, before any numbers are computed. Step 8 needs it.

Writes `cov_block_sets2can_discon_com_4pts_and_optypes.npy` and `cov_block2TEB_block2can_sn_alm_info2nterms.npy` to `cov_dir`.

In [ ]:
sbatch("7b", "--mem 64G --cpus-per-task 8 --time 30:00",
       "get_ana_cov_blocks_recipe.py")
print()
show_outputs(cov_dir, "cov_block")

---
## 8: analytic covariance

Schematically

$$\mathrm{Var}(C_\ell) \sim \frac{2}{(2\ell+1)\, f_{\rm sky}^{\rm eff}}\left(C_\ell + N_\ell\right)^2,
\qquad f_{\rm sky}^{\rm eff} = \frac{w_2^2}{w_4}\, f_{\rm sky}$$

with the signal from step 2, the noise from step 6 and the window geometry from step 7. The signal term is cosmic variance. The general case couples spectra,

$$\mathrm{Cov}\!\left(C_\ell^{ij}, C_{\ell'}^{pq}\right) \supset
C_\ell^{ip} C_{\ell'}^{jq} + C_\ell^{iq} C_{\ell'}^{jp} + (\text{noise terms})$$

which is where the bookkeeping over arrays and splits comes from. Spectra that share a map have correlated errors: two arrays give 6 blocks, 3 diagonal and 3 off-diagonal. The number of blocks grows like $N^4/4$.

Writes `analytic_cov_<spec_ij>_<spec_pq>.npy` to `cov_dir`, one per pair of spectra.

In [ ]:
sbatch("8", "--mem 200G --cpus-per-task 16 --time 60:00", "get_ana_cov_blocks.py")

In [ ]:
show_outputs(cov_dir, "analytic_cov")

cov = np.load(opj(cov_dir, f"analytic_cov_{spec_name}_{spec_name}.npy"))
print(f"\nshape {cov.shape}  (nbin {nbin})")

In [ ]:
# as a correlation matrix the diagonal divides out, leaving just the bin-to-bin coupling
cov_TT = cov[:nbin, :nbin] if cov.shape[0] > nbin else cov
sig = np.sqrt(np.diag(cov_TT))
corr = cov_TT / np.outer(sig, sig)

fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
im = axs[0].imshow(corr, origin="lower", cmap="RdBu_r", vmin=-0.5, vmax=0.5)
axs[0].set_xlabel("bin"); axs[0].set_ylabel("bin"); axs[0].set_title("bin correlation")
plt.colorbar(im, ax=axs[0], fraction=0.045)

axs[1].plot(lb[:len(sig)], sig, "o-", ms=3)
axs[1].set_yscale("log"); axs[1].set_xlabel(r"$\ell$")
axs[1].set_ylabel(r"$\sigma(D_\ell)$"); axs[1].grid(alpha=0.3)
fig.tight_layout()

In [ ]:
# with more than one spectrum the covariance couples them. the full matrix is
# (nspec*nbin) square; blocks off the diagonal say how much two spectra move together.
full_fn = opj(cov_dir, f"analytic_cov_{spec_names[0]}_{spec_names[-1]}.npy")
if os.path.exists(full_fn) and len(spec_names) > 1:
    cross_block = np.load(full_fn)
    print(f"{spec_names[0]}\n  x {spec_names[-1]}\n  -> {cross_block.shape}")

    b = cross_block[:nbin, :nbin]
    dg = np.sqrt(np.abs(np.diag(b)))
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    im = ax.imshow(b / np.outer(dg, dg), origin="lower", cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xlabel("bin"); ax.set_ylabel("bin")
    ax.set_title("TT correlation between the two autos")
    plt.colorbar(im, ax=ax, fraction=0.045)
else:
    print("no cross-spectrum covariance block found")

---
## 9: foreground covariance

Marginalising over the foreground parameters $\theta$ adds a correction to the step 8 block (it does not replace it):

$$\mathsf{C}^{\rm fg} = \frac{\partial D_\ell^{\rm fg}}{\partial \theta}\,\mathsf{C}_\theta\,
\frac{\partial D_{\ell'}^{\rm fg}}{\partial \theta}^{\!\top}$$

The script needs either `p_act_chain_filename` (an ACT chain) or `fg_covmat_path` (a precomputed foreground parameter covariance). The paramfiles here set neither, so this step is not run in this notebook.

Writes `fg_marginalization_cov_<na>x<nb>_<nc>x<nd>.npy` to `cov_dir`.

In [ ]:
sbatch("9", "--mem 64G --cpus-per-task 8 --time 30:00", "get_fg_covariance_blocks.py")
print()
show_outputs(cov_dir, "fg_marginalization")

---
## Result

All spectra against theory, with residuals: points should scatter within about $1\sigma$ and $\chi^2$/dof should be near 1.

$$\sigma_b = \sqrt{\mathsf{C}_{bb}}, \qquad r_b = \frac{D_b - D_b^{\rm th}}{\sigma_b}$$

The diagonal $\chi^2 = \sum_b r_b^2$ ignores the correlation between bins. The full version is $\chi^2 = r^{\top}\mathsf{C}^{-1} r$, with $r$ stacking all spectra and $\mathsf{C}$ including the off-diagonal blocks.

In [ ]:
# read all three spectra and their diagonal covariance blocks
res = {}
for sn in spec_names:
    _, ps = so_spectra.read_ps(opj(spec_dir, f"{spec_type}_{sn}_cross.dat"), spectra=spectra)
    c = np.load(opj(cov_dir, f"analytic_cov_{sn}_{sn}.npy"))
    c = c[:nbin, :nbin] if c.shape[0] > nbin else c
    n = c.shape[0]
    res[sn] = dict(lb=lb[:n], Db=ps["TT"][:n], cov=c, sig=np.sqrt(np.diag(c)))

th = {sn: np.interp(r["lb"], l_th, ps_th["TT"]) for sn, r in res.items()}

fig, axs = plt.subplots(2, 1, figsize=(9, 7), sharex=True,
                        gridspec_kw={"height_ratios": [2, 1]})
axs[0].plot(l_th, ps_th["TT"], "k-", lw=1, alpha=0.6, label="theory")
for sn, r in res.items():
    lbl = sn.replace(f"{sv}_", "").replace("x", " x ")
    axs[0].errorbar(r["lb"], r["Db"], yerr=r["sig"], fmt="o", ms=3, capsize=2, label=lbl)
axs[0].set_yscale("log"); axs[0].set_ylabel(r"$D_\ell^{TT}$ [$\mu K^2$]")
axs[0].legend(fontsize=8); axs[0].grid(alpha=0.3)

axs[1].axhline(0, color="k", lw=1)
axs[1].axhspan(-1, 1, color="grey", alpha=0.2)
for sn, r in res.items():
    axs[1].plot(r["lb"], (r["Db"] - th[sn]) / r["sig"], "o", ms=3)
axs[1].set_xlim(0, 6000); axs[1].set_ylim(-5, 5)
axs[1].set_xlabel(r"$\ell$"); axs[1].set_ylabel(r"(data-theory)/$\sigma$")
axs[1].grid(alpha=0.3)
fig.tight_layout()

In [ ]:
# chi2 per spectrum, diagonal errors only, over the range where theory is trustworthy
LO, HI = 500, 4000
print(f"{'spectrum':32s} {'bins':>5s} {'chi2/dof':>9s} {'mean r':>8s} {'scatter':>8s}")
for sn, r in res.items():
    m = (r["lb"] > LO) & (r["lb"] < HI)
    rr = ((r["Db"] - th[sn]) / r["sig"])[m]
    print(f"{sn.replace(sv + '_', ''):32s} {m.sum():5d} "
          f"{np.sum(rr**2) / m.sum():9.2f} {rr.mean():8.2f} {rr.std():8.2f}")

In [ ]:
# joint chi2 using the full block covariance. stack the residuals, assemble the matrix
# including the off-diagonal blocks, invert. this is the number to quote.
ns = len(spec_names)
m = (res[spec_names[0]]["lb"] > LO) & (res[spec_names[0]]["lb"] < HI)
nb_sel = m.sum()

r_stack = np.concatenate([((res[sn]["Db"] - th[sn]) / 1.0)[m] for sn in spec_names])
C = np.zeros((ns * nb_sel, ns * nb_sel))

for i, sa in enumerate(spec_names):
    for j, sb in enumerate(spec_names):
        fn = opj(cov_dir, f"analytic_cov_{sa}_{sb}.npy")
        if not os.path.exists(fn):
            fn = opj(cov_dir, f"analytic_cov_{sb}_{sa}.npy")
        if not os.path.exists(fn):
            print(f"missing block {sa} x {sb}")
            continue
        blk = np.load(fn)
        blk = blk[:nbin, :nbin] if blk.shape[0] > nbin else blk
        blk = blk[np.ix_(m, m)]
        C[i*nb_sel:(i+1)*nb_sel, j*nb_sel:(j+1)*nb_sel] = blk if fn.endswith(f"{sa}_{sb}.npy") else blk.T

C = 0.5 * (C + C.T)                      # symmetrise against rounding
chi2_full = r_stack @ np.linalg.solve(C, r_stack)
dof = len(r_stack)

# same residuals, diagonal only, for comparison
chi2_diag = np.sum(r_stack**2 / np.diag(C))

print(f"bins per spectrum {nb_sel}, spectra {ns}, dof {dof}")
print(f"chi2/dof  diagonal only : {chi2_diag / dof:.2f}")
print(f"chi2/dof  full covariance: {chi2_full / dof:.2f}")
print(f"condition number {np.linalg.cond(C):.2e}")

---
## Notes

- Error bars in the plots are `sqrt(diag(cov))`. Use the full matrix for a fit.
- The windows come from the i1 coverage only, so they are wider than in a run that intersects all twelve arrays, and the numbers will not match a full run.
- Scaling with the number of arrays $N$: masks and windows like $N$, spectra and mode coupling matrices like $N(N+1)/2$, covariance blocks like $N^4/4$. To widen the scope edit `arrays_lat_iso` in the paramfile, set `--ntasks` to the new number of spectra and rerun from 0c.